In [15]:
# Setup & Sampling
import json, time, numpy as np, pandas as pd, torch
from pathlib import Path
from sentence_transformers import SentenceTransformer

PROCESSED_FILE = Path("../data/processed/cicids_processed.csv")
OUTPUT_DIR = Path("../data/embeddings")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SAMPLE_FILE = OUTPUT_DIR / "cicids_sample.csv"
EMBED_FILE = OUTPUT_DIR / "cicids_embeddings.npy"
META_FILE = OUTPUT_DIR / "cicids_embedding_meta.json"

MODEL_NAME = "all-MiniLM-L6-v2"
RANDOM_SEED = 42
SAMPLE_PER_TACTIC = 2000
BENIGN_SAMPLE = 2000
BATCH_SIZE = 128

df = pd.read_csv(PROCESSED_FILE, index_col="alert_id", low_memory=False)
attack_parts = []
attack_df = df[df["attck_tactic"] != "Benign"].copy()
for tactic in sorted(attack_df["attck_tactic"].dropna().unique()):
    subset = attack_df[attack_df["attck_tactic"] == tactic]
    n = min(len(subset), SAMPLE_PER_TACTIC)
    attack_parts.append(subset.sample(n=n, random_state=RANDOM_SEED))

benign_sample = df[df["attck_tactic"] == "Benign"].sample(n=min(len(df[df["attck_tactic"]=="Benign"]), BENIGN_SAMPLE), random_state=RANDOM_SEED)
sample = pd.concat(attack_parts + [benign_sample], ignore_index=False)
sample["alert_id"] = sample.index
sample = sample.sample(frac=1, random_state=RANDOM_SEED).reset_index(drop=True)
sample.index.name = "sample_id"
sample["sample_id"] = sample.index
sample = sample[sample["alert_text"].notna()].copy()
sample.to_csv(SAMPLE_FILE, index=True)
print(f"Final sample: {len(sample)} rows")

Final sample: 10684 rows


In [16]:
# Embed & Save
device = "cuda" if torch.cuda.is_available() else "cpu"
model = SentenceTransformer(MODEL_NAME, device=device)
texts = sample["alert_text"].tolist()

start = time.time()
embeddings = model.encode(texts, batch_size=BATCH_SIZE, show_progress_bar=True, convert_to_numpy=True, normalize_embeddings=True)
print(f"Embedding complete in {time.time()-start:.1f}s | Shape: {embeddings.shape}")

np.save(EMBED_FILE, embeddings)
meta = {"model": MODEL_NAME, "device": device, "rows": len(sample), "dim": embeddings.shape[1], "seed": RANDOM_SEED}
with open(META_FILE, "w") as f:
    json.dump(meta, f, indent=2)
print(f"Saved to {EMBED_FILE} and {META_FILE}")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Batches:   0%|          | 0/84 [00:00<?, ?it/s]

Embedding complete in 5.0s | Shape: (10684, 384)
Saved to ../data/embeddings/cicids_embeddings.npy and ../data/embeddings/cicids_embedding_meta.json
